### ¿Qué origen tiene más
### delitos sexuales per cápita?

#### 1️⃣ Delitos sexuales 2025

In [26]:
import pandas as pd

df = pd.read_csv(
  '../datasets/'
  'delicuencia_nacionalidad.csv',
  sep=';',
  dtype={'Total': str}
)

df['Total'] = pd.to_numeric(
  df['Total']
  .str.replace('.', '',
    regex=False),
  errors='coerce'
)

# Categoría 8: contra la libertad
# e indemnidad sexuales
es_sexual = (
  df['Tipo de Delito: Nivel 2']
  .str.contains('sexual',
    case=False, na=False)
)
# Subtotal (sin bajar a Nivel 3)
es_sub = (
  df['Tipo de Delito: Nivel 3']
  .isna()
)
es_2025 = df['Periodo'] == 2025

delitos = (
  df[es_sexual & es_sub & es_2025]
  .set_index('Nacionalidad')
  ['Total']
)

delitos

Nacionalidad
Total                              6111.0
Española                           4131.0
País de la UE28 sin España            NaN
País de la UE27_2020 sin España     320.0
País de Europa menos UE28             NaN
País de Europa menos UE27_2020       52.0
De Africa                           651.0
De América                          844.0
De Asia                             113.0
De Oceanía                            0.0
Name: Total, dtype: float64

#### 2️⃣ Cargamos la población

In [27]:
pob = pd.read_csv(
  '../datasets/'
  'evolucion_migracion.csv',
  sep=';'
)

pob['Total'] = pd.to_numeric(
  pob['Total']
  .str.replace('.', '',
    regex=False),
  errors='coerce'
)

pob = pob[
  pob['Periodo']
  == '1 de enero de 2025'
]

#### 3️⃣ Cálculo normalizado según origen

In [29]:
total = pob[
  (pob['Sexo'] == 'Total')
  & (pob['Grupo quinquenal de edad']
     == 'Todas las edades')
].set_index('Nacionalidad')['Total']

# En DELITOS, América viene junta
def delitos_cont(s):
  ue = 'País de la UE27_2020 sin España'
  eu2 = 'País de Europa menos UE27_2020'
  return {
    'Española': s['Española'],
    'Europa': s[ue] + s[eu2],
    'África': s['De Africa'],
    'América': s['De América'],
    'Asia': s['De Asia'],
  }

# En POBLACIÓN, América va partida
def pob_cont(p):
  ue = 'País de la UE27_2020 sin España'
  eu2 = 'País de Europa menos UE27_2020'
  am = (
    p['De América del Norte']
    + p['De Centro América y Caribe']
    + p['De Sudamérica']
  )
  return {
    'Española': p['Española'],
    'Europa': p[ue] + p[eu2],
    'África': p['De Africa'],
    'América': am,
    'Asia': p['De Asia'],
  }


d = delitos_cont(delitos)
p_tot = pob_cont(total)

# 1 delito por cada X habitantes
factor = pd.Series({
  k: p_tot[k] / d[k]
  for k in d
}).sort_values()

# Factor vs España
tabla = pd.DataFrame({
  '1 de cada': factor,
  'factor': factor['Española'] / factor,
})

print(tabla.round(1))

          1 de cada  factor
África       2100.7     4.9
América      2988.2     3.4
Asia         4984.9     2.1
Europa       6583.8     1.6
Española    10219.4     1.0
